# Chapter 6 — Atmospheric Effects — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 13. Atmospheric Refraction, Radio Horizon & Absorption — Ch 6 *(future, outdoor)*

Indoors these are negligible; they matter only for **long outdoor links** (outdoor city
track) and **mmWave** absorption. Extends §0's radio horizon with the real refraction model.

- Refractivity `N = (77.6/T)(P + 4810 e/T)` (eq 6.4); gradient `dN/dh = −(Ns/H) e^(−h/H)`,
  H=7 km. → `refractivity()`, `refractivity_gradient()`.
- Effective-earth factor `k = 1/(1 + r·dn/dh)` (eq 6.11); radio horizon `d ≈ √(2 k r h)`.
  → `k_factor()`, `radio_horizon_km()`. **Ducting** when `dn/dh = −157×10⁻⁶ /km` (k → ∞).
- Gaseous absorption `A = γ·d` (dB), `γ = γ_O2 + γ_H2O`. Lines: **22 GHz (water vapor),
  60 GHz (oxygen, ~15 dB/km)**; ~0.05 dB/km at 1 GHz. Radar → ×2. → `atmospheric_loss_db()`.


In [ ]:
def refractivity(P_mb, e_mb, T_K):
    return (77.6/T_K)*(P_mb + 4810*e_mb/T_K)                 # eq 6.4

def refractivity_gradient(Ns, h_km, H_km=7.0):
    return -Ns/H_km*np.exp(-h_km/H_km)                        # eq 6.7

def k_factor(dNdh_per_km, r_earth_km=6370.0):
    return 1.0/(1 + r_earth_km*dNdh_per_km*1e-6)              # eq 6.11

def radio_horizon_km(h_m, k=4/3, r_earth_km=6370.0):
    return np.sqrt(2*k*r_earth_km*h_m/1000.0)                 # d ~ sqrt(2 k r h)

def atmospheric_loss_db(gamma_db_per_km, d_km, radar=False):
    return gamma_db_per_km*d_km*(2 if radar else 1)           # A = gamma*d (x2 for radar)

# Example 6.1: 50 m tower at 2 km ASL; P=1100 mb, e=12 mb, T=260 K
Ns = refractivity(1100, 12, 260)
dNdh = refractivity_gradient(Ns, 2.0)
k = k_factor(dNdh)
print(f"Ex 6.1: N={Ns:.1f} (book 394.6), dN/dh={dNdh:.2f} (-42.36), k={k:.3f} (1.370), "
      f"horizon={radio_horizon_km(50, k):.1f} km (book 29.5)")
